# Domain-Adversarial BERT - Full Dataset Training with Cross-Validation

This notebook trains a binary domain-adversarial BERT model using the same overall
workflow as `New_FYP_7_1-3.ipynb`, but swaps the baseline classifier for a DANN-style
model.

## Scope for this first implementation
1. Binary source-vs-target domain classification.
2. One source dataset per notebook cell.
3. Initial runs for WELFake and FakeNewsNet only.
4. Cross-validation is run on the source dataset, while the target dataset is used as
   an unlabeled domain stream during training.

## Workflow
1. Imports and environment setup.
2. Dataset paths and source-target pair configuration.
3. Dataset classes for standard evaluation and paired domain-adversarial training.
4. DANN model with gradient reversal.
5. Custom Trainer for label loss + domain loss.
6. Helper functions for metrics, device selection, saving, and loading.
7. Cross-validation on the source dataset with paired target batches.
8. Cross-dataset evaluation on datasets not used in the current run.
9. One execution cell per source dataset.


# Section 1: Imports & Setup

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import BertModel, BertTokenizer
from transformers import EarlyStoppingCallback, Trainer, TrainerCallback, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

from google.colab import drive

drive.mount('/content/drive')

try:
    import torch_xla as torch_xla_pkg
    import torch_xla.core.xla_model as xm
    if not hasattr(torch, 'xla'):
        torch.xla = torch_xla_pkg
    _TORCH_XLA_AVAILABLE = True
except Exception:
    xm = None
    _TORCH_XLA_AVAILABLE = False

SEED = random.randint(0, 4294967295)
print(f'Random seed: {SEED}')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Section 2: Dataset Paths

In [ ]:
DATASETS = {
    'WELFake': '/content/drive/MyDrive/datasets/WELFake_processed.csv',
    'FakeNewsNet': '/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv',
    'Fake_News_Detection': '/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv',
    'ISOT': '/content/drive/MyDrive/datasets/ISOT_processed.csv',
    'Fake_News_Classification': '/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv',
}

RUN_PAIRS = {
    'WELFake': 'FakeNewsNet',
    'FakeNewsNet': 'WELFake',
}


# Section 3: Dataset Classes

In [ ]:
def _tensor_at(values, idx):
    return torch.tensor(values[idx], dtype=torch.long)


class FakeNewsDataset(Dataset):
    """Standard dataset for validation, testing, and cross-dataset evaluation."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
        )
        item = {key: torch.tensor(val, dtype=torch.long) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


class DANNPairDataset(Dataset):
    """Pairs a labeled source example with a dynamically random unlabeled target example."""

    def __init__(self, source_texts, source_labels, target_texts, tokenizer, max_length=128):
        self.source_texts = source_texts
        self.source_labels = source_labels
        self.target_texts = target_texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.source_len = len(source_labels)
        self.target_len = len(target_texts)

    def __getitem__(self, idx):
        import random
        target_idx = random.randint(0, self.target_len - 1)
        
        # Tokenize source text on the fly
        source_text = self.source_texts[idx]
        source_enc = self.tokenizer(
            source_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
        )
        item = {key: torch.tensor(val, dtype=torch.long) for key, val in source_enc.items()}
        item['labels'] = torch.tensor(self.source_labels[idx], dtype=torch.long)
        
        # Tokenize target text on the fly
        target_text = self.target_texts[target_idx]
        target_enc = self.tokenizer(
            target_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
        )
        for key, val in target_enc.items():
            item[f'{key}_target'] = torch.tensor(val, dtype=torch.long)
            
        return item

    def __len__(self):
        return self.source_len


# Section 4: DANN BERT Model

In [ ]:
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, lambda_domain):
        ctx.lambda_domain = lambda_domain
        return inputs.view_as(inputs)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_domain * grad_output, None


def grad_reverse(inputs, lambda_domain):
    return GradientReversalFn.apply(inputs, lambda_domain)


class DANNBERT(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1, domain_hidden_dim=256):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.label_head = nn.Linear(768, num_labels)
        self.domain_head = nn.Sequential(
            nn.Linear(768, domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(domain_hidden_dim, 2),
        )

    def gradient_checkpointing_enable(self, **kwargs):
        self.bert.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def encode(self, input_ids=None, attention_mask=None, token_type_ids=None):
        if token_type_ids is not None:
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        else:
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]

    def classify(self, features):
        return self.label_head(self.dropout(features))

    def domain_classify(self, features, lambda_domain=1.0):
        reversed_features = grad_reverse(features, lambda_domain)
        return self.domain_head(reversed_features)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, return_features=False, **kwargs):
        features = self.encode(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = self.classify(features)
        if return_features:
            return logits, features
        return SequenceClassifierOutput(logits=logits)


# Section 5: Custom Trainer

In [ ]:
class LambdaSchedulerCallback(TrainerCallback):
    def __init__(self, max_lambda=1.0, warmup_ratio=0.1):
        self.max_lambda = max_lambda
        self.warmup_ratio = warmup_ratio
        self.trainer = None

    def on_step_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get('trainer') or self.trainer
        if trainer is None or not hasattr(trainer, 'lambda_domain_tensor'):
            return
        total_steps = state.max_steps
        if total_steps <= 0:
            return
        current_step = state.global_step
        
        # Standard DANN sigmoid-like scheduling formula
        p = current_step / total_steps
        new_lambda = self.max_lambda * (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)
        trainer.lambda_domain_tensor.fill_(new_lambda)


class DANNTrainer(Trainer):
    def __init__(self, *args, lambda_domain=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_domain_tensor = torch.tensor(
            lambda_domain,
            device=self.args.device,
            dtype=torch.float,
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = self._prepare_inputs(inputs)
        labels = inputs['labels']
        token_type_ids = inputs.get('token_type_ids')

        if 'input_ids_target' not in inputs:
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=token_type_ids,
            )
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
            loss = nn.CrossEntropyLoss()(logits, labels)
            return (loss, outputs) if return_outputs else loss

        source_features = model.encode(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=token_type_ids,
        )
        source_logits = model.classify(source_features)

        target_token_type_ids = inputs.get('token_type_ids_target')
        target_features = model.encode(
            input_ids=inputs['input_ids_target'],
            attention_mask=inputs['attention_mask_target'],
            token_type_ids=target_token_type_ids,
        )

        batch_size = labels.size(0)
        domain_features = torch.cat([source_features, target_features], dim=0)
        domain_logits = model.domain_classify(domain_features, lambda_domain=self.lambda_domain_tensor)
        domain_labels = torch.cat([
            torch.zeros(batch_size, dtype=torch.long, device=domain_logits.device),
            torch.ones(batch_size, dtype=torch.long, device=domain_logits.device),
        ], dim=0)

        label_loss = nn.CrossEntropyLoss()(source_logits, labels)
        domain_loss = nn.CrossEntropyLoss()(domain_logits, domain_labels)
        # Standard DANN loss summation: GRL already handles the -lambda gradient scaling
        total_loss = label_loss + domain_loss

        if return_outputs:
            outputs = SequenceClassifierOutput(logits=source_logits)
            return total_loss, outputs
        return total_loss


# Section 6: Helper Functions

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='binary',
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }


def load_device():
    if _TORCH_XLA_AVAILABLE and xm is not None:
        try:
            device = xm.xla_device()
            print(f'Using TPU: {device}')
            return device, True
        except Exception:
            pass

    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'Using CUDA: {torch.cuda.get_device_name(0)}')
    else:
        device = torch.device('cpu')
        print('Using CPU (training will be slow)')
    return device, False


def save_model(model, tokenizer, output_path, use_tpu, device):
    print('\n' + '=' * 60)
    print('MODEL SAVING')
    print('=' * 60)

    os.makedirs(output_path, exist_ok=True)

    if use_tpu:
        model.to('cpu')
        print('Moved model to CPU for saving.')

    torch.save(model.state_dict(), os.path.join(output_path, 'dann_bert.pt'))
    tokenizer.save_pretrained(output_path)
    print(f'Model saved to {output_path}')

    if use_tpu:
        model.to(device)
        print('Moved model back to TPU.')

    print('\n' + '=' * 60)
    print('DOMAIN-ADVERSARIAL BERT COMPLETE')
    print('=' * 60)


def load_saved_model(model_dir, device=None):
    if device is None:
        device, _ = load_device()
    model = DANNBERT()
    state_dict = torch.load(os.path.join(model_dir, 'dann_bert.pt'), map_location='cpu')
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    return model, tokenizer


def _slice_encodings(encodings, indices):
    return {key: [values[i] for i in indices] for key, values in encodings.items()}


# Section 7: Adversarial Cross-Validation

In [ ]:
def cross_validate_dann(
    source_train_texts,
    source_train_labels,
    target_train_texts,
    source_test_dataset,
    tokenizer,
    compute_metrics_fn,
    use_tpu,
    device,
    n_splits=5,
):
    print('\n' + '=' * 60)
    print(f'STARTING {n_splits}-FOLD CROSS VALIDATION (DANN BERT)')
    print('=' * 60)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(source_train_labels))

    fold_accuracies = []
    fold_f1_scores = []
    fold_test_results = []

    best_f1 = -1.0
    best_model_state = None
    best_fold_idx = -1

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, source_train_labels)):
        print(f'\n--- FOLD {fold + 1} ---')

        train_texts = [source_train_texts[i] for i in train_idx]
        val_texts = [source_train_texts[i] for i in val_idx]
        train_labels = [source_train_labels[i] for i in train_idx]
        val_labels = [source_train_labels[i] for i in val_idx]

        train_dataset = DANNPairDataset(train_texts, train_labels, target_train_texts, tokenizer)
        val_dataset = FakeNewsDataset(val_texts, val_labels, tokenizer)

        model = DANNBERT(num_labels=2, dropout=0.1)
        if not use_tpu:
            model.to(device)

        training_kwargs = {
            'output_dir': f'./results_dann_fold_{fold + 1}',
            'num_train_epochs': 3,
            'per_device_train_batch_size': 16,
            'per_device_eval_batch_size': 32,
            'gradient_accumulation_steps': 2,
            'save_strategy': 'steps',
            'save_steps': 128,
            'save_total_limit': 1,
            'bf16': use_tpu,
            'gradient_checkpointing': not use_tpu,
            'report_to': 'none',
            'optim': 'adamw_torch',
            'logging_dir': './logs',
            'logging_steps': 128,
            'metric_for_best_model': 'f1',
            'load_best_model_at_end': True,
            'weight_decay': 0.01,
            'remove_unused_columns': False,
            'label_names': ['labels'],
        }
        if 'evaluation_strategy' in TrainingArguments.__init__.__code__.co_varnames:
            training_kwargs['evaluation_strategy'] = 'steps'
            training_kwargs['eval_steps'] = 128
        else:
            training_kwargs['eval_strategy'] = 'steps'
            training_kwargs['eval_steps'] = 128

        training_args = TrainingArguments(**training_kwargs)

        lambda_cb = LambdaSchedulerCallback(max_lambda=1.0, warmup_ratio=0.1)
        trainer = DANNTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics_fn,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=3),
                lambda_cb,
            ],
            lambda_domain=0.0,
        )
        lambda_cb.trainer = trainer

        trainer.train()

        eval_metrics = trainer.evaluate()
        val_f1 = eval_metrics['eval_f1']
        fold_f1_scores.append(val_f1)
        fold_accuracies.append(eval_metrics['eval_accuracy'])
        print(f"Fold {fold + 1} Validation - Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_fold_idx = fold

        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            test_loader = DataLoader(source_test_dataset, batch_size=32)
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                token_type_ids = batch.get('token_type_ids')
                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)
                labels = batch['labels']

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                )
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
                preds = logits.argmax(-1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(labels.numpy())

        precision, recall, f1, _ = precision_recall_fscore_support(
            all_targets,
            all_preds,
            average='binary',
        )
        acc = accuracy_score(all_targets, all_preds)
        test_metrics = {
            'eval_accuracy': acc,
            'eval_f1': f1,
            'eval_precision': precision,
            'eval_recall': recall,
        }
        fold_test_results.append(test_metrics)
        print(
            f"Fold {fold + 1} Test     - Accuracy: {acc:.4f}, F1: {f1:.4f}, "
            f"Precision: {precision:.4f}, Recall: {recall:.4f}"
        )

        del model, trainer, train_dataset, val_dataset
        import gc
        gc.collect()
        if not use_tpu and torch.cuda.is_available():
            torch.cuda.empty_cache()

    print('\n' + '=' * 60)
    print('CROSS-VALIDATION SUMMARY')
    print('=' * 60)
    print(f"{'Fold':<6} {'Val Acc':<10} {'Val F1':<10} {'Test Acc':<10} {'Test F1':<10} {'Test Prec':<10} {'Test Rec':<10}")
    print('-' * 66)
    for i in range(len(fold_test_results)):
        metrics = fold_test_results[i]
        print(
            f"{i + 1:<6} {fold_accuracies[i]:<10.4f} {fold_f1_scores[i]:<10.4f} "
            f"{metrics['eval_accuracy']:<10.4f} {metrics['eval_f1']:<10.4f} "
            f"{metrics['eval_precision']:<10.4f} {metrics['eval_recall']:<10.4f}"
        )

    avg_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    avg_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)

    print(f'\nAverage Validation Accuracy: {avg_acc:.4f} (Std Dev: {std_acc:.4f})')
    print(f'Average Validation F1-score: {avg_f1:.4f} (Std Dev: {std_f1:.4f})')

    best_test = fold_test_results[best_fold_idx]
    print(f"\nBest Fold: {best_fold_idx + 1} (Val F1: {fold_f1_scores[best_fold_idx]:.4f})")
    print(
        f"  Test Results - Accuracy: {best_test['eval_accuracy']:.4f}, "
        f"F1: {best_test['eval_f1']:.4f}, Precision: {best_test['eval_precision']:.4f}, "
        f"Recall: {best_test['eval_recall']:.4f}"
    )

    best_model = DANNBERT(num_labels=2, dropout=0.1)
    best_model.load_state_dict(best_model_state)
    if not use_tpu:
        best_model.to(device)

    return best_model


# Section 8: Cross-Dataset Evaluation

In [ ]:
def cross_dataset_evaluation(
    model,
    tokenizer,
    source_dataset_name,
    all_datasets_paths,
    compute_metrics_fn,
    excluded_datasets=None,
):
    excluded = set(excluded_datasets or [])
    excluded.add(source_dataset_name)

    device, use_tpu = load_device()
    print('\n' + '!' * 60)
    print(f'CROSS-DATASET GENERALIZATION: {source_dataset_name}')
    print('!' * 60)

    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model

        def forward(self, input_ids, attention_mask, labels=None, token_type_ids=None, **kwargs):
            outputs = self.inner_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            return SequenceClassifierOutput(loss=loss, logits=logits)

    eval_model = ModelWrapper(model)
    results = {}

    for name, path in all_datasets_paths.items():
        if name in excluded:
            continue

        print(f'\nTesting on unseen dataset: {name} (Full Dataset)...')
        df = pd.read_csv(path).dropna().reset_index(drop=True)
        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        dataset = FakeNewsDataset(test_texts, test_labels, tokenizer)

        eval_trainer = Trainer(
            model=eval_model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir='./temp_eval',
                remove_unused_columns=False,
                label_names=['labels'],
                per_device_eval_batch_size=32,
                report_to='none',
            ),
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        print(
            f"  -> {name} Accuracy: {metrics.get('eval_accuracy', 0):.4f}, "
            f"F1: {metrics.get('eval_f1', 0):.4f}, "
            f"Precision: {metrics.get('eval_precision', 0):.4f}, "
            f"Recall: {metrics.get('eval_recall', 0):.4f}"
        )
        # Free memory immediately to avoid RAM leakage
        del dataset, eval_trainer, df, test_texts, test_labels
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results


# Section 9: Main Training Loop

In [ ]:
def dann_train_loop(source_dataset_name, target_dataset_name, output_path, n_splits=5):
    source_dataset_path = DATASETS[source_dataset_name]
    target_dataset_path = DATASETS[target_dataset_name]

    print(f'\nInitialising DANN experiment: {source_dataset_name} -> {target_dataset_name}')

    source_df = pd.read_csv(source_dataset_path).dropna().reset_index(drop=True)
    target_df = pd.read_csv(target_dataset_path).dropna().reset_index(drop=True)
    print(f"Source rows: {len(source_df)}")
    print(f"Target rows: {len(target_df)}")
    print(f"Source label distribution:\n{source_df['label'].value_counts()}")

    source_texts = source_df['combined_text'].tolist()
    source_labels = source_df['label'].tolist()
    target_texts = target_df['combined_text'].tolist()

    print('\n' + '=' * 60)
    print('TRAIN / TEST SPLIT (85-15) ON SOURCE DOMAIN')
    print('=' * 60)

    indices = list(range(len(source_df)))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.15,
        random_state=42,
        stratify=source_labels,
    )

    source_train_texts = [source_texts[i] for i in train_idx]
    source_train_labels = [source_labels[i] for i in train_idx]
    source_test_texts = [source_texts[i] for i in test_idx]
    source_test_labels = [source_labels[i] for i in test_idx]

    print(f'Source training pool: {len(source_train_texts)}')
    print(f'Source test set: {len(source_test_texts)}')

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    source_test_dataset = FakeNewsDataset(source_test_texts, source_test_labels, tokenizer)

    device, use_tpu = load_device()
    best_model = cross_validate_dann(
        source_train_texts,
        source_train_labels,
        target_texts,
        source_test_dataset,
        tokenizer,
        compute_metrics,
        use_tpu,
        device,
        n_splits=n_splits,
    )

    save_model(best_model, tokenizer, output_path, use_tpu, device)

    cross_dataset_evaluation(
        best_model,
        tokenizer,
        source_dataset_name,
        DATASETS,
        compute_metrics,
        # Evaluate on the adapted target dataset as well to measure Adaptation Success!
        excluded_datasets={source_dataset_name},
    )


def dann_single_fold_experiment(
    source_name='WELFake',
    test_name='FakeNewsNet',
    target_names=None,
    output_path='/content/drive/MyDrive/models/DANN_BERT_SingleFold_WELFake_to_FakeNewsNet_MultiTarget',
):
    if target_names is None:
        target_names = ['Fake_News_Detection', 'ISOT', 'Fake_News_Classification']

    print(f'\nInitialising Single Fold Multi-Target Experiment:')
    print(f'  Source (Labeled): {source_name}')
    print(f'  Test (Evaluation): {test_name}')
    print(f'  Targets (Unlabeled): {target_names}')

    # Load Source Dataset
    source_dataset_path = DATASETS[source_name]
    source_df = pd.read_csv(source_dataset_path).dropna().reset_index(drop=True)
    source_texts = source_df['combined_text'].tolist()
    source_labels = source_df['label'].tolist()

    # Load and Concatenate Targets
    target_texts = []
    for name in target_names:
        path = DATASETS[name]
        df = pd.read_csv(path).dropna().reset_index(drop=True)
        target_texts.extend(df['combined_text'].tolist())
    print(f"Source rows: {len(source_df)}")
    print(f"Combined Target rows: {len(target_texts)}")

    # Split Source into 85% Train, 15% Val
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        source_texts,
        source_labels,
        test_size=0.15,
        random_state=42,
        stratify=source_labels,
    )
    print(f'Source training pool: {len(train_texts)}')
    print(f'Source validation pool: {len(val_texts)}')

    # Load Test Dataset
    test_dataset_path = DATASETS[test_name]
    test_df = pd.read_csv(test_dataset_path).dropna().reset_index(drop=True)
    test_texts = test_df['combined_text'].tolist()
    test_labels = test_df['label'].tolist()
    print(f"Test rows ({test_name}): {len(test_df)}")

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    train_dataset = DANNPairDataset(train_texts, train_labels, target_texts, tokenizer)
    val_dataset = FakeNewsDataset(val_texts, val_labels, tokenizer)
    test_dataset = FakeNewsDataset(test_texts, test_labels, tokenizer)

    device, use_tpu = load_device()
    model = DANNBERT(num_labels=2, dropout=0.1)
    if not use_tpu:
        model.to(device)

    training_kwargs = {
        'output_dir': './results_dann_single_fold',
        'num_train_epochs': 3,
        'per_device_train_batch_size': 16,
        'per_device_eval_batch_size': 32,
        'gradient_accumulation_steps': 2,
        'save_strategy': 'steps',
        'save_steps': 128,
        'save_total_limit': 1,
        'bf16': use_tpu,
        'gradient_checkpointing': not use_tpu,
        'report_to': 'none',
        'optim': 'adamw_torch',
        'logging_dir': './logs',
        'logging_steps': 128,
        'metric_for_best_model': 'f1',
        'load_best_model_at_end': True,
        'weight_decay': 0.01,
        'remove_unused_columns': False,
        'label_names': ['labels'],
    }
    if 'evaluation_strategy' in TrainingArguments.__init__.__code__.co_varnames:
        training_kwargs['evaluation_strategy'] = 'steps'
        training_kwargs['eval_steps'] = 128
    else:
        training_kwargs['eval_strategy'] = 'steps'
        training_kwargs['eval_steps'] = 128

    training_args = TrainingArguments(**training_kwargs)

    lambda_cb = LambdaSchedulerCallback(max_lambda=1.0, warmup_ratio=0.1)
    trainer = DANNTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            lambda_cb,
        ],
        lambda_domain=0.0,
    )
    lambda_cb.trainer = trainer

    print('Training single-fold DANN model...')
    trainer.train()

    eval_metrics = trainer.evaluate()
    print(f"Source Validation - Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1: {eval_metrics['eval_f1']:.4f}")

    print(f'Evaluating on Test Dataset ({test_name})...')
    eval_model = model
    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model

        def forward(self, input_ids, attention_mask, labels=None, token_type_ids=None, **kwargs):
            outputs = self.inner_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            return SequenceClassifierOutput(loss=loss, logits=logits)

    eval_trainer = Trainer(
        model=ModelWrapper(eval_model),
        compute_metrics=compute_metrics,
        args=TrainingArguments(
            output_dir='./temp_eval',
            remove_unused_columns=False,
            label_names=['labels'],
            per_device_eval_batch_size=32,
            report_to='none',
        ),
    )
    test_metrics = eval_trainer.evaluate(eval_dataset=test_dataset)
    print(
        f"Test Results ({test_name}) -> Accuracy: {test_metrics.get('eval_accuracy', 0):.4f}, "
        f"F1: {test_metrics.get('eval_f1', 0):.4f}, "
        f"Precision: {test_metrics.get('eval_precision', 0):.4f}, "
        f"Recall: {test_metrics.get('eval_recall', 0):.4f}"
    )

    save_model(model, tokenizer, output_path, use_tpu, device)

    cross_dataset_evaluation(
        model,
        tokenizer,
        source_name,
        DATASETS,
        compute_metrics,
        excluded_datasets={source_name, test_name},
    )


# WELFake Source Run

In [ ]:
dann_train_loop(
    'WELFake',
    'FakeNewsNet',
    '/content/drive/MyDrive/models/DANN_BERT_WELFake_to_FakeNewsNet',
    n_splits=5,
)


# FakeNewsNet Source Run

In [ ]:
dann_train_loop(
    'FakeNewsNet',
    'WELFake',
    '/content/drive/MyDrive/models/DANN_BERT_FakeNewsNet_to_WELFake',
    n_splits=5,
)


# Single-Fold Multi-Target Experiment

In [ ]:
dann_single_fold_experiment(
    source_name='WELFake',
    test_name='FakeNewsNet',
    target_names=['Fake_News_Detection', 'ISOT', 'Fake_News_Classification'],
    output_path='/content/drive/MyDrive/models/DANN_BERT_SingleFold_WELFake_to_FakeNewsNet_MultiTarget',
)
